# 11. Content-Based Filtering — TF-IDF + Cosine Similarity <a id="11-content-based" name="11-content-based"></a>

In [ ]:
def fuzzy_match(title, title_to_idx):
    """Find best match for a title (case-insensitive, partial match)."""
    tl = title.lower().strip()
    if tl in title_to_idx.index:
        return tl
    # Partial match
    matches = [t for t in title_to_idx.index if tl in t or t in tl]
    if matches:
        return sorted(matches, key=lambda x: abs(len(x) - len(tl)))[0]
    # Token match
    for tok in tl.split():
        if len(tok) > 3:
            matches = [t for t in title_to_idx.index if tok in t]
            if matches:
                return sorted(matches, key=lambda x: abs(len(x)-len(tl)))[0]
    return None

def content_recommendations(title, n=5, genre_filter=None,
                              min_year=None, max_runtime=None,
                              min_rating=None, language=None):
    """
    Content-based recommendations using TF-IDF cosine similarity.

    Parameters:
    -----------
    title        : Input movie title
    n            : Number of recommendations (default 5)
    genre_filter : Filter by genre (optional)
    min_year     : Minimum release year (optional)
    max_runtime  : Maximum runtime in minutes (optional)
    min_rating   : Minimum TMDB rating (optional)
    language     : Filter by original language (optional)
    """
    matched = fuzzy_match(title, title_to_idx)
    if matched is None:
        print(f"Movie not found: '{title}'")
        return None

    idx = title_to_idx[matched]
    if isinstance(idx, pd.Series): idx = idx.iloc[0]

    # Cosine similarity
    sim = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()

    scored = df.copy()
    scored["_sim"] = sim
    scored = scored[scored.index != idx]

    # Apply filters
    if genre_filter:
        scored = scored[scored["genres"].str.contains(genre_filter, case=False, na=False)]
    if min_year:
        scored = scored[scored["release_year"] >= min_year]
    if max_runtime:
        scored = scored[scored["runtime"] <= max_runtime]
    if min_rating:
        scored = scored[scored["vote_average"] >= min_rating]
    if language:
        scored = scored[scored["original_language"] == language]

    results = scored.nlargest(n, "_sim")[
        ["title","release_year","primary_genre","vote_average","vote_count",
         "runtime","director","_sim","poster_path"]
    ].reset_index(drop=True)
    results.index += 1
    results.columns = ["Title","Year","Genre","Rating","Votes","Runtime (min)","Director","Similarity","Poster"]
    return results, df.iloc[idx]

print(" Content-based recommender ready")


 Content-based recommender ready


In [ ]:
#  DEMO : 10 test cases
test_cases = [
    ("Star Wars",          {}),
    ("Forrest Gump",       {"min_rating": 7.0}),
    ("The Dark Knight",    {"genre_filter": "Action"}),
    ("Inception",          {"min_year": 2010, "max_runtime": 130}),
    ("Finding Nemo",       {"genre_filter": "Animation"}),
    ("Pulp Fiction",       {"genre_filter": "Crime"}),
    ("The Godfather",      {"min_rating": 8.0}),
    ("Interstellar",       {"genre_filter": "Science Fiction"}),
    ("Toy Story",          {"max_runtime": 100}),
    ("The Silence of the Lambs", {"genre_filter": "Thriller"}),
]

for movie, filters in test_cases:
    print(f"\n{'='*65}")
    print(f" Input: '{movie}'  |  Filters: {filters or 'None'}")
    print(f"{'='*65}")
    result = content_recommendations(movie, n=5, **filters)
    if result:
        recs, inp = result
        print(f"  Because you liked: {inp['title']} ({int(inp['release_year'])} |  {inp['vote_average']:.1f})")
        print(recs.drop(columns=["Poster"]).to_string())


 Input: 'Star Wars'  |  Filters: None
  Because you liked: Star Wars (1977 |  8.2)
                                          Title  Year      Genre  Rating  Votes  Runtime (min)          Director  Similarity
1                       The Empire Strikes Back  1980  Adventure    8.39  18286            124    Irvin Kershner        0.48
2                            Return of the Jedi  1983  Adventure    7.91  16826            132  Richard Marquand        0.40
3                  Star Wars: The Force Awakens  2015  Adventure    7.25  20441            136       J.J. Abrams        0.36
4                      Star Wars: The Last Jedi  2017  Adventure    6.75  16238            152      Rian Johnson        0.30
5  Star Wars: Episode III - Revenge of the Sith  2005  Adventure    7.46  14834            140      George Lucas        0.28

 Input: 'Forrest Gump'  |  Filters: {'min_rating': 7.0}
  Because you liked: Forrest Gump (1994 |  8.5)
                    Title  Year   Genre  Rating  Votes  Runti